# Pipeline 2 — Capilares (YOLO11s-seg + tiles 3×3)

- Dataset: `dataset_v3.1_yolo11_tiled_3x3` (fora do repo)
- Classe: só **Capilar** via `classes=[0]` (ignora `microcotiledone`)
- Mesma família/aug do pipeline 1; calibração de área nativa (pixel isotrópico)


In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

# Force temp/cache off C: — always use D: for this project
_TMP_D = Path(r'D:\projeto_placentas_clayton\temp_ml')
_TMP_D.mkdir(parents=True, exist_ok=True)
os.environ['TEMP'] = str(_TMP_D)
os.environ['TMP'] = str(_TMP_D)
os.environ['TMPDIR'] = str(_TMP_D)

# Ultralytics global dirs must be absolute on D: (relative paths follow CWD and can hit C:)
_ULTRA_SETTINGS = Path(os.environ.get('APPDATA', '')) / 'Ultralytics' / 'settings.json'
if _ULTRA_SETTINGS.is_file():
    _cfg = json.loads(_ULTRA_SETTINGS.read_text(encoding='utf-8'))
    _cfg['datasets_dir'] = r'D:\projeto_placentas_clayton\datasets_ultralytics'
    _cfg['weights_dir'] = r'D:\projeto_placentas_clayton\weights_ultralytics'
    _cfg['runs_dir'] = r'D:\projeto_placentas_clayton\runs_ultralytics'
    _ULTRA_SETTINGS.write_text(json.dumps(_cfg, indent=2), encoding='utf-8')
    print('Ultralytics dirs pinned to D:')
    print(' ', _cfg['datasets_dir'])
    print(' ', _cfg['weights_dir'])
    print(' ', _cfg['runs_dir'])

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

print(f'TEMP/TMP -> {_TMP_D}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

Ultralytics dirs pinned to D:
  D:\projeto_placentas_clayton\datasets_ultralytics
  D:\projeto_placentas_clayton\weights_ultralytics
  D:\projeto_placentas_clayton\runs_ultralytics
TEMP/TMP -> D:\projeto_placentas_clayton\temp_ml
Ultralytics: 8.4.54
PyTorch: 2.5.1+cu121
GPU: NVIDIA GeForce GTX 1650 SUPER


In [2]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v2').exists():
            return p
        p = p.parent
    raise RuntimeError('Repo root not found (.git + v2)')

REPO_ROOT = discover_repo_root()

CFG = {
    'data_yaml': 'v3_capilar_yolo11s/data_capilar_tiled.yaml',
    'pretrained': 'yolo11s-seg.pt',  # na raiz do repo ou cwd
    'project': 'v3_capilar_yolo11s/runs',
    'run_name': 'capilar_yolo11s_tiled_v1',
    'epochs': 120,
    'imgsz': 640,
    'batch': 2,            # GTX 1650 SUPER 4GB
    'patience': 30,
    'max_det': 150,        # valid: max ~56 capilares/tile
    'classes': [0],        # Capilar only
    'iou_match': 0.5,
    # Native isotropic px (bar 50um measured as 72px on stretched-640 X-axis):
    # m = (50/72)*(640/4140) um/px  ->  AREA_FACTOR = m**2
    'area_factor': (50 / 72) ** 2 * (640 / 4140) ** 2,
    'conf_candidates': [round(x, 2) for x in np.arange(0.15, 0.71, 0.02)],
    'output_root': 'v3_capilar_yolo11s/artifacts',
}

data_yaml = (REPO_ROOT / CFG['data_yaml']).resolve()
out_root = (REPO_ROOT / CFG['output_root']).resolve()
out_bench = out_root / 'benchmarks'
out_reports = out_root / 'reports'
for p in (out_root, out_bench, out_reports):
    p.mkdir(parents=True, exist_ok=True)

with data_yaml.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

dataset_root = Path(data_cfg['path'])
val_images = (dataset_root / data_cfg['val']).resolve()
val_labels = (dataset_root / 'valid' / 'labels').resolve()

pretrained = Path(CFG['pretrained'])
if not pretrained.is_file():
    pretrained = (REPO_ROOT / CFG['pretrained']).resolve()

print('repo:', REPO_ROOT)
print('data_yaml:', data_yaml)
print('dataset:', dataset_root)
print('val_images:', val_images)
print('pretrained:', pretrained, 'exists=', pretrained.is_file())
print('area_factor (um2/px2):', CFG['area_factor'])
print('n val tiles:', len(list(val_images.glob('*.jpg'))))

repo: D:\projeto_placentas_clayton\dev\projeto-placentas
data_yaml: D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\data_capilar_tiled.yaml
dataset: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3
val_images: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3\valid\images
pretrained: D:\projeto_placentas_clayton\dev\projeto-placentas\yolo11s-seg.pt exists= True
area_factor (um2/px2): 0.011524823461313616
n val tiles: 243


## 1) Treino

Mesma receita do pipeline 1 (`retina_masks`, aug forte). Rode esta célula na máquina com GPU.


In [3]:
DO_TRAIN = True  # False para pular e ir direto ao sweep com um best.pt já treinado

if DO_TRAIN:
    model = YOLO(str(pretrained))
    train_results = model.train(
        data=str(data_yaml),
        epochs=CFG['epochs'],
        imgsz=CFG['imgsz'],
        batch=CFG['batch'],
        patience=CFG['patience'],
        device=0 if torch.cuda.is_available() else 'cpu',
        project=str((REPO_ROOT / CFG['project']).resolve()),
        name=CFG['run_name'],
        exist_ok=True,
        classes=CFG['classes'],
        max_det=CFG['max_det'],
        # segmentation quality
        retina_masks=True,
        overlap_mask=False,
        mask_ratio=1,
        # augmentation (mesmo espírito do pipeline 1)
        degrees=90.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        scale=0.5,
        hsv_s=0.7,
    )
    best_pt = Path(train_results.save_dir) / 'weights' / 'best.pt'
else:
    best_pt = (REPO_ROOT / CFG['project'] / CFG['run_name'] / 'weights' / 'best.pt').resolve()

print('best.pt:', best_pt)
assert best_pt.is_file(), f'Checkpoint não encontrado: {best_pt}'

New https://pypi.org/project/ultralytics/8.4.143 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.54  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=[0], close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\data_capilar_tiled.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=1, max_det=150, mixup=0.1, mod

d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 21.823.8 ms, read: 8.06.4 MB/s, size: 134.1 KB)
val: Scanning D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3\valid\labels.cache... 243 images, 1 backgrounds, 242 corrupt: 100% ━━━━━━━━━━━━ 243/243  0.0s
val: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3\valid\images\ROSILHA-M-B_011_jpg.rf.4a966541cdff84d474cd4516dd35b742_r0c0.jpg: ignoring corrupt image/label: could not convert string to float: '0,3181159420'
val: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3\valid\images\ROSILHA-M-B_011_jpg.rf.4a966541cdff84d474cd4516dd35b742_r0c1.jpg: ignoring corrupt image/label: could not convert string to float: '0,0644927536'
val: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3\valid\images\ROSILHA-M-B_011_j

d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\ultralytics\utils\metrics.py:877: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      2/120      1.63G          0          0      37.15          0          0          1        640: 100% ━━━━━━━━━━━━ 7/7 5.2it/s 1.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 13.4it/s 0.1s
                   all          1          0          0          0          0          0          0          0          0          0
WARNING no labels found in segment set, cannot compute metrics without labels


...


      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
    120/120      1.63G          0          0     0.1904          0          0          1        640: 100% ━━━━━━━━━━━━ 7/7 4.7it/s 1.5s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 25.4it/s 0.0s
                   all          1          0          0          0          0          0          0          0          0          0
WARNING no labels found in segment set, cannot compute metrics without labels


d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\ultralytics\utils\metrics.py:877: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



120 epochs completed in 0.108 hours.
Optimizer stripped from D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\runs\capilar_yolo11s_tiled_v1\weights\last.pt, 20.5MB
Optimizer stripped from D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\runs\capilar_yolo11s_tiled_v1\weights\best.pt, 20.5MB

Validating D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\runs\capilar_yolo11s_tiled_v1\weights\best.pt...
Ultralytics 8.4.54  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
YOLO11s-seg summary (fused): 114 layers, 10,067,590 parameters, 0 gradients, 32.8 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 12.6it/s 0.1s


d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\ultralytics\utils\metrics.py:695: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\ultralytics\utils\metrics.py:741: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\ultralytics\utils\metrics.py:741: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value enc

                   all          1          0          0          0          0          0          0          0          0          0
WARNING no labels found in segment set, cannot compute metrics without labels
Speed: 1.3ms preprocess, 74.2ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\runs\capilar_yolo11s_tiled_v1
best.pt: D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\runs\capilar_yolo11s_tiled_v1\weights\best.pt


## 2) Helpers de métrica (tile-level)

Avaliação por **tile** (igual ao split de validação). Agregação por imagem-fonte + merge de borda fica para o próximo notebook.


In [4]:
CAPILAR_CLS = 0


def parse_gt_masks(label_path: Path, img_w: int, img_h: int, cls_keep: int = CAPILAR_CLS):
    masks = []
    if not label_path.exists():
        return masks
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        if cls != cls_keep:
            continue
        # tolerate accidental locale commas in labels
        coords = np.array([float(x.replace(',', '.')) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= img_w
        coords[:, 1] *= img_h
        m = np.zeros((img_h, img_w), dtype=np.uint8)
        cv2.fillPoly(m, [coords.astype(np.int32)], 1)
        masks.append(m)
    return masks


def masks_from_result(r):
    if r.masks is None:
        return []
    # r.masks.data is already mapped to original image size when retina_masks=True
    return [(m > 0.5).astype(np.uint8) for m in r.masks.data.cpu().numpy()]


def iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union else 0.0


def greedy_match(pred_masks, gt_masks, thr: float):
    pairs = []
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            s = iou(pm, gm)
            if s >= thr:
                pairs.append((s, i, j))
    pairs.sort(reverse=True)
    used_p, used_g, matched = set(), set(), []
    for s, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        matched.append((s, i, j))
    return matched


def eval_at_conf(model: YOLO, conf: float):
    img_paths = sorted(val_images.glob('*.jpg'))
    tp = fp = fn = 0
    ious = []
    gt_area = 0
    pred_area = 0

    for img_path in img_paths:
        im = cv2.imread(str(img_path))
        h, w = im.shape[:2]
        lbl = val_labels / f'{img_path.stem}.txt'
        gt_masks = parse_gt_masks(lbl, w, h)

        r = model.predict(
            source=str(img_path),
            conf=conf,
            imgsz=CFG['imgsz'],
            retina_masks=True,
            max_det=CFG['max_det'],
            classes=CFG['classes'],
            verbose=False,
        )[0]
        pred_masks = masks_from_result(r)

        # ensure mask spatial size == image size
        fixed = []
        for m in pred_masks:
            if m.shape[0] != h or m.shape[1] != w:
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            fixed.append(m)
        pred_masks = fixed

        matches = greedy_match(pred_masks, gt_masks, CFG['iou_match'])
        tp += len(matches)
        fp += len(pred_masks) - len(matches)
        fn += len(gt_masks) - len(matches)
        ious.extend([s for s, _, _ in matches])

        gt_area += int(sum(int(m.sum()) for m in gt_masks))
        pred_area += int(sum(int(m.sum()) for m in pred_masks))

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    area_rel_err = abs(pred_area - gt_area) / gt_area if gt_area else 0.0
    return {
        'conf': conf,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'mean_iou': mean_iou,
        'gt_area_px': gt_area,
        'pred_area_px': pred_area,
        'gt_area_um2': gt_area * CFG['area_factor'],
        'pred_area_um2': pred_area * CFG['area_factor'],
        'area_rel_error': area_rel_err,
        'score': 0.5 * f1 + 0.3 * mean_iou + 0.2 * (1.0 - area_rel_err),
    }

print('helpers ok')

helpers ok


## 3) Confidence sweep


In [5]:
eval_model = YOLO(str(best_pt))
rows = []
t0 = time.time()
for conf in CFG['conf_candidates']:
    row = eval_at_conf(eval_model, conf)
    rows.append(row)
    print(
        f"conf={conf:.2f}  F1={row['f1']:.4f}  IoU={row['mean_iou']:.4f}  "
        f"area_err={row['area_rel_error']:.4f}  score={row['score']:.4f}"
    )

sweep_df = pd.DataFrame(rows).sort_values('score', ascending=False)
sweep_path = out_bench / 'validation_conf_sweep.csv'
sweep_df.to_csv(sweep_path, index=False)

best = sweep_df.iloc[0].to_dict()
selected = {
    'model': 'yolo11s-seg',
    'run_name': CFG['run_name'],
    'checkpoint': str(best_pt),
    'best_conf': float(best['conf']),
    'f1': float(best['f1']),
    'mean_iou': float(best['mean_iou']),
    'area_rel_error': float(best['area_rel_error']),
    'weighted_score': float(best['score']),
    'area_factor_um2_per_px2': CFG['area_factor'],
    'classes': CFG['classes'],
    'notes': 'Tile-level metrics on Capilar only; field-level merge TBD',
}
(out_bench / 'selected_confidence.json').write_text(json.dumps(selected, indent=2), encoding='utf-8')

print('\n=== BEST ===')
print(json.dumps(selected, indent=2))
print(f'sweep saved: {sweep_path}')
print(f'elapsed: {time.time() - t0:.1f}s')

ValueError: could not convert string to float: '0,8376811594'

## 4) Relatório por tile no melhor conf


In [ ]:
best_conf = float(selected['best_conf'])
totals = []
instances = []

for img_path in sorted(val_images.glob('*.jpg')):
    im = cv2.imread(str(img_path))
    h, w = im.shape[:2]
    gt_masks = parse_gt_masks(val_labels / f'{img_path.stem}.txt', w, h)
    r = eval_model.predict(
        source=str(img_path),
        conf=best_conf,
        imgsz=CFG['imgsz'],
        retina_masks=True,
        max_det=CFG['max_det'],
        classes=CFG['classes'],
        verbose=False,
    )[0]
    pred_masks = masks_from_result(r)
    pred_masks = [
        cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST) if m.shape[:2] != (h, w) else m
        for m in pred_masks
    ]
    matches = greedy_match(pred_masks, gt_masks, CFG['iou_match'])
    matched_p = {i for _, i, _ in matches}
    matched_g = {j for _, _, j in matches}

    gt_a = int(sum(int(m.sum()) for m in gt_masks))
    pr_a = int(sum(int(m.sum()) for m in pred_masks))
    totals.append({
        'Tile': img_path.name,
        'GT_Count': len(gt_masks),
        'AI_Count': len(pred_masks),
        'Matched': len(matches),
        'FP': len(pred_masks) - len(matches),
        'FN': len(gt_masks) - len(matches),
        'GT_Area_px': gt_a,
        'AI_Area_px': pr_a,
        'GT_Area_um2': gt_a * CFG['area_factor'],
        'AI_Area_um2': pr_a * CFG['area_factor'],
        'Area_Diff_pct': ((pr_a - gt_a) / gt_a * 100.0) if gt_a else 0.0,
    })

    for s, i, j in matches:
        ga = int(gt_masks[j].sum()); pa = int(pred_masks[i].sum())
        instances.append({
            'Tile': img_path.name, 'Match_Type': 'TP', 'AI_Index': i, 'GT_Index': j,
            'IoU': s, 'GT_Area_px': ga, 'AI_Area_px': pa,
            'GT_Area_um2': ga * CFG['area_factor'], 'AI_Area_um2': pa * CFG['area_factor'],
        })
    for i, pm in enumerate(pred_masks):
        if i in matched_p:
            continue
        pa = int(pm.sum())
        instances.append({
            'Tile': img_path.name, 'Match_Type': 'FP', 'AI_Index': i, 'GT_Index': -1,
            'IoU': 0.0, 'GT_Area_px': 0, 'AI_Area_px': pa,
            'GT_Area_um2': 0.0, 'AI_Area_um2': pa * CFG['area_factor'],
        })
    for j, gm in enumerate(gt_masks):
        if j in matched_g:
            continue
        ga = int(gm.sum())
        instances.append({
            'Tile': img_path.name, 'Match_Type': 'FN', 'AI_Index': -1, 'GT_Index': j,
            'IoU': 0.0, 'GT_Area_px': ga, 'AI_Area_px': 0,
            'GT_Area_um2': ga * CFG['area_factor'], 'AI_Area_um2': 0.0,
        })

totals_df = pd.DataFrame(totals)
inst_df = pd.DataFrame(instances)
totals_path = out_reports / 'capilar_tile_totals_report.csv'
inst_path = out_reports / 'capilar_tile_instance_report.csv'
totals_df.to_csv(totals_path, index=False)
inst_df.to_csv(inst_path, index=False)
print(totals_df[['GT_Count', 'AI_Count', 'Area_Diff_pct']].describe())
print('saved:', totals_path)
print('saved:', inst_path)